# 06 — Real-data setup and audit

This notebook inspects a dataset produced by `scripts/prepare_btc_lob.py`. Training remains in scripts.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROCESSED_DIR = Path('../data/processed/btc_h40_cost')
PROCESSED_DIR

In [ ]:
metadata_path = PROCESSED_DIR / 'metadata.json'
if not metadata_path.exists():
    print('Prepare the raw BTC dataset first. See docs/REAL_DATA_PIPELINE.md')
    metadata = None
else:
    metadata = json.loads(metadata_path.read_text())
    display(pd.Series(metadata, dtype='object'))

In [ ]:
if metadata is not None:
    labels = np.load(PROCESSED_DIR / 'labels.npy', mmap_mode='r')
    splits = np.load(PROCESSED_DIR / 'splits.npz')
    rows = []
    for split_name in ('train', 'validation', 'test'):
        values = labels[splits[split_name]]
        counts = np.bincount(values, minlength=3)
        rows.append({'split': split_name, 'down/short': counts[0], 'flat': counts[1], 'up/long': counts[2]})
    display(pd.DataFrame(rows).set_index('split'))

In [ ]:
if metadata is not None:
    source = np.load(PROCESSED_DIR / 'source_features.npy', mmap_mode='r')
    timestamps = np.load(PROCESSED_DIR / 'timestamps.npy')
    ask = source[:, 0]
    bid = source[:, 2]
    mid = (ask + bid) / 2
    spread_bps = (ask - bid) / mid * 10_000
    audit = pd.DataFrame({
        'timestamp': pd.to_datetime(timestamps),
        'midprice': mid,
        'spread_bps': spread_bps,
    })
    display(audit.describe(include='all'))

In [ ]:
if metadata is not None:
    sample = audit.iloc[: min(5000, len(audit))]
    ax = sample.plot(x='timestamp', y='midprice', title='Mid-price sample', figsize=(11, 4))
    ax.set_ylabel('Price')
    plt.show()

In [ ]:
if metadata is not None:
    names = json.loads((PROCESSED_DIR / 'tabular_feature_names.json').read_text())
    tabular = np.load(PROCESSED_DIR / 'tabular_features.npy', mmap_mode='r')
    display(pd.DataFrame(tabular[:5], columns=names))

## Audit checklist

- Confirm snapshot frequency and gaps.
- Confirm no crossed books.
- Inspect duplicate-timestamp policy.
- Check class balance by day.
- Check spread and volatility regimes.
- Confirm costs and label horizon match the planned backtest.